RAG
notes:
- conda env: rag-ta, requirements-rag.txt

# Import Library

In [ ]:
import json
import os
from typing import List, Dict, Any, Tuple, Optional
import numpy as np


from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from sentence_transformers import SentenceTransformer

from openai import OpenAI
from pymilvus import Collection, AnnSearchRequest, WeightedRanker,  connections, CollectionSchema, FieldSchema, DataType, utility
from pymilvus import IndexType, MetricType, Index


from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain_community.chat_message_histories import ChatMessageHistory


# Initiate variable

In [36]:
dataset_path = "dataset_korupsi_json"

model_name = "gpt-4o-mini"
temperature = 0.1

# Data Preparation

## Load Dataset XML to JSON

In [37]:
# import os
# import xml.etree.ElementTree as ET
# import json
# from tqdm import tqdm

# # Fungsi untuk mengonversi elemen XML ke dictionary
# def xml_to_dict(element):
#     result = {}
#     # Tambahkan atribut elemen ke dictionary
#     result.update(element.attrib)
#     # Tambahkan elemen anak ke dictionary
#     for child in element:
#         if len(child):  # Jika elemen memiliki anak
#             result[child.tag] = xml_to_dict(child)
#         else:  # Jika elemen tidak memiliki anak
#             result[child.tag] = child.text.strip() if child.text else None
#     return result

# # Fungsi untuk mengonversi file XML ke JSON dengan progress bar
# def convert_xml_to_json(input_folder, output_folder):
#     # Pastikan folder output ada
#     if not os.path.exists(output_folder):
#         os.makedirs(output_folder)

#     # Dapatkan daftar file XML di folder input
#     xml_files = [f for f in os.listdir(input_folder) if f.endswith(".xml")]

#     # Inisialisasi progress bar
#     with tqdm(total=len(xml_files), desc="Processing Files", unit="file") as pbar:
#         for filename in xml_files:
#             file_path = os.path.join(input_folder, filename)
#             try:
#                 # Parse file XML
#                 tree = ET.parse(file_path)
#                 root = tree.getroot()

#                 # Konversi ke dictionary
#                 data_dict = xml_to_dict(root)

#                 # Nama file JSON
#                 json_filename = os.path.splitext(filename)[0] + ".json"
#                 json_path = os.path.join(output_folder, json_filename)

#                 # Simpan sebagai file JSON
#                 with open(json_path, 'w', encoding='utf-8') as json_file:
#                     json.dump(data_dict, json_file, indent=4, ensure_ascii=False)

#                 pbar.set_postfix({"Current File": filename})
#             except Exception as e:
#                 pbar.set_postfix({"Error": str(e)})
            
#             # Update progress bar
#             pbar.update(1)


In [38]:
# # Folder input dan output
# input_folder = "/Users/auva/Documents/GitHub/indo-law-rag/dataset"  # Ganti dengan path folder input
# output_folder = "/Users/auva/Documents/GitHub/indo-law-rag/dataset_json"  # Ganti dengan path folder output

# # Jalankan konversi
# convert_xml_to_json(input_folder, output_folder)

## Load JSON to List

In [39]:
# def load_json_documents(folder_path: str) -> List[Document]:
#     """
#     Load all JSON files from a folder recursively using JSONLoader
    
#     Args:
#         folder_path (str): Path to the folder containing JSON files
        
#     Returns:
#         List[Document]: List of loaded documents
#     """
#     all_documents = []
    
#     # Walk through all files in directory and subdirectories
#     for root, dirs, files in os.walk(folder_path):
#         for file in files:
#             if file.endswith('.json'):
#                 file_path = os.path.join(root, file)
#                 try:
#                     # Initialize JSONLoader for each file
#                     loader = JSONLoader(
#                         file_path=file_path,
#                         jq_schema='.',  # You might need to adjust this based on your JSON structure
#                         text_content=False
#                     )
                    
#                     # Load documents from the file
#                     documents = loader.load()
#                     all_documents.extend(documents)
                    
#                     print(f"Successfully loaded: {file_path}")
#                 except Exception as e:
#                     print(f"Error loading {file_path}: {str(e)}")
    
#     return all_documents


In [40]:
def load_json_documents(directory_path: str) -> List[Dict[str, Any]]:
    """
    Memuat semua file JSON dari direktori yang ditentukan
    """
    documents = []
    for filename in os.listdir(directory_path):
        if filename.endswith('.json'):
            with open(os.path.join(directory_path, filename), 'r', encoding='utf-8') as f:
                documents.append(json.load(f))
    return documents


In [41]:
documents = load_json_documents(dataset_path)
# Print total number of documents loaded
print(f"\nTotal documents loaded: {len(documents)}")


Total documents loaded: 97


In [42]:
documents[0]

{'identitas': '1 nama lengkap mochtar hidayat bin h soekarno pranoto 2 tempat lahir semarang 55 tahun 19 mei 1960 laki laki indonesia 3 umur tanggal lahir 4 jenis kelamin 5 kebangsaan 6 tempat tinggal jl tusam raya no 20 rt 001 rw 001\nkel pedalangan kec banyumanik kota semarang\nislam wiraswasta\n7 agama 8 pekerjaan 9 pendidikan d 3 akuntansi terdakwa ditahan dalam tahanan rumah tahanan negara oleh\n1 penyidik tidak ditahan',
 'riwayat_perkara': 'terdakwa didampingi oleh penasihat hukum dr h umar maruf sh sp n m hum m fajar subhi a k arif sh mh devi rivaldi sh rudini hasyim rado s h anang purwono s h dari kantor advokat pengacara umar fajar rekan yang beralamat di jl majapahit ruko gayamsari no 61 kota semarang berdasarkan surat kuasa khusus tanggal 13 mei 2016\npengadilan tindak pidana korupsi tersebut\nsetelah membaca\npenetapan ketua pengadilan tindak pidana korupsi pada pengadilan negeri semarang nomor 68 pen pid sus tpk 2016 pn smg tanggal 10 mei 2016 tentang penunjukan majelis h

## Chunking Docs

In [43]:
# Definisikan ukuran chunk dan overlap
MAX_CHUNK_SIZE = 1000  # karakter
OVERLAP_SIZE = 200  # karakter

In [44]:
def extract_document_info(document: Dict[str, Any]) -> Dict[str, str]:
    """
    Ekstrak informasi penting dari dokumen untuk metadata
    """
    # Ekstrak nama terdakwa dari bagian identitas
    nama_terdakwa = ""
    if "identitas" in document:
        # Konversi ke string untuk memastikan re.search bekerja dengan benar
        identitas_str = str(document["identitas"])
        match = re.search(r"nama lengkap ([^\n]+)", identitas_str, re.IGNORECASE)
        if match:
            nama_terdakwa = match.group(1).strip()
    
    # Nomor perkara bisa diambil dari URL atau bagian riwayat
    nomor_perkara = ""
    if "url" in document:
        # Extract dari URL jika ada
        nomor_perkara = os.path.basename(document["url"]).replace(".html", "")
    
    return {
        "nama_terdakwa": nama_terdakwa,
        "nomor_perkara": nomor_perkara,
        "url": document.get("url", "")
    }

In [45]:
def chunk_long_text(text: str, section: str, document_info: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Chunking untuk teks panjang dengan penanda paragraf spesifik
    """
    chunks = []
    
    # Pilih penanda paragraf berdasarkan bagian
    if section == "pertimbangan_hukum":
        # Split berdasarkan "menimbang bahwa"
        paragraphs = re.split(r'(menimbang bahwa)', text, flags=re.IGNORECASE)
        # Gabungkan kembali penanda dengan paragrafnya
        processed_paragraphs = []
        for i in range(0, len(paragraphs)-1, 2):
            if i+1 < len(paragraphs):
                processed_paragraphs.append(paragraphs[i] + paragraphs[i+1])
            else:
                processed_paragraphs.append(paragraphs[i])
        
        if len(processed_paragraphs) == 0:  # Jika tidak ada split yang berhasil
            processed_paragraphs = [text]
    
    elif section == "amar_putusan":
        # Split berdasarkan angka urutan dalam amar putusan (misalnya "1.", "2.")
        paragraphs = re.split(r'\n\d+\.', text)
        processed_paragraphs = paragraphs
    
    else:
        # Default chunking dengan ukuran tetap jika tidak ada penanda khusus
        current_pos = 0
        processed_paragraphs = []
        
        while current_pos < len(text):
            end_pos = min(current_pos + MAX_CHUNK_SIZE, len(text))
            
            # Jika kita tidak di akhir teks, cari batas kalimat terdekat
            if end_pos < len(text):
                # Coba temukan akhir kalimat terdekat (titik, tanda tanya, tanda seru)
                sentence_end = max(
                    text.rfind('. ', current_pos, end_pos),
                    text.rfind('? ', current_pos, end_pos),
                    text.rfind('! ', current_pos, end_pos)
                )
                
                if sentence_end != -1:
                    end_pos = sentence_end + 1  # +1 untuk menyertakan tanda baca
            
            processed_paragraphs.append(text[current_pos:end_pos])
            
            # Pindah ke posisi berikutnya dengan overlap
            current_pos = end_pos - OVERLAP_SIZE if end_pos < len(text) else end_pos
    
    # Buat chunk dari paragraf yang diproses
    current_chunk = ""
    chunk_index = 0
    
    for paragraph in processed_paragraphs:
        # Jika paragraf ini akan membuat chunk terlalu besar
        if len(current_chunk) + len(paragraph) > MAX_CHUNK_SIZE and current_chunk:
            # Simpan chunk saat ini
            chunks.append({
                "text": current_chunk,
                "metadata": {
                    "document_type": "putusan_pengadilan",
                    "section": section,
                    "chunk_index": chunk_index,
                    "nama_terdakwa": document_info["nama_terdakwa"],
                    "nomor_perkara": document_info["nomor_perkara"],
                    "url": document_info["url"]
                }
            })
            chunk_index += 1
            current_chunk = paragraph
        else:
            # Tambahkan paragraf ke chunk saat ini
            if current_chunk:
                current_chunk += " "
            current_chunk += paragraph
    
    # Jangan lupa chunk terakhir
    if current_chunk:
        chunks.append({
            "text": current_chunk,
            "metadata": {
                "document_type": "putusan_pengadilan",
                "section": section,
                "chunk_index": chunk_index,
                "nama_terdakwa": document_info["nama_terdakwa"],
                "nomor_perkara": document_info["nomor_perkara"],
                "url": document_info["url"]
            }
        })
    
    return chunks


In [46]:
def chunk_by_structure(document: Dict[str, Any], document_info: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Chunking berdasarkan struktur dokumen (field utama)
    """
    chunks = []
    
    # Iterasi melalui field utama dokumen
    for field, content in document.items():
        if field in ["identitas", "riwayat_perkara", "amar_putusan", "pertimbangan_hukum"]:
            # Untuk field pendek, simpan sebagai satu chunk
            if len(content) <= MAX_CHUNK_SIZE:
                chunks.append({
                    "text": content,
                    "metadata": {
                        "document_type": "putusan_pengadilan",
                        "section": field,
                        "nama_terdakwa": document_info["nama_terdakwa"],
                        "nomor_perkara": document_info["nomor_perkara"],
                        "url": document_info["url"]
                    }
                })
            # Untuk field panjang, lakukan chunking tambahan
            else:
                section_chunks = chunk_long_text(content, field, document_info)
                chunks.extend(section_chunks)
    
    return chunks

In [47]:
def process_documents(directory_path: str, output_path: str):
    """
    Proses semua dokumen dan simpan hasil chunking
    """
    documents = load_json_documents(directory_path)
    all_chunks = []
    
    for doc in documents:
        doc_info = extract_document_info(doc)
        doc_chunks = chunk_by_structure(doc, doc_info)
        all_chunks.extend(doc_chunks)
    
    # Simpan semua chunk ke file JSON
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)
    
    print(f"Processed {len(documents)} documents into {len(all_chunks)} chunks.")


In [48]:
process_documents("./dataset_korupsi_json", "./chunks_korupsi.json")

Processed 97 documents into 5520 chunks.


In [52]:
# # text splitter
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=800, #max num of char 
#     chunk_overlap=100,#overlap between chunks to maintain context 
#     length_function=len,
#     is_separator_regex=False,
# )

# chunk_docs = text_splitter.split_documents(documents)

## Create Embeddings

In [ ]:
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:
def create_embedding(text: str) -> List[float]:
    """
    Membuat embedding vektor dari teks menggunakan OpenAI API
    """
    response = openai_client.embeddings.create(
        input=text,
        model="text-embedding-3-small"  # Anda bisa menggunakan model embedding lain
    )
    return response.data[0].embedding

# Define LLM

In [53]:
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI

llm = OpenAI(model=model_name, temperature=temperature)

In [54]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
embedding_dim = 1536

# Milvus Preparation

## Connect to Milvus

In [55]:
def connect_to_milvus():
    """Connect to Milvus standalone server"""
    try:
        connections.connect(
            alias="default",
            host='127.0.0.1',
            port='19530',
            db_name='database_ta'
        )
        print("Successfully connected to Milvus")
    except Exception as e:
        print(f"Failed to connect to Milvus: {e}")
        raise
    
connect_to_milvus()

Successfully connected to Milvus


In [56]:
# only run once, for creating the db 
# create database 
# database = db.create_database("database_ta_testing")

In [57]:
from pymilvus import FieldSchema, DataType, CollectionSchema, Collection, Index
from tqdm import tqdm
import json

## Create Schema

In [58]:
def create_legal_document_schema():
    """Create schema for legal document collection"""
    
    # Define fields
    id_field = FieldSchema(
        name="id", 
        dtype=DataType.INT64, 
        is_primary=True, 
        description="primary id"
    )
    
    content_vector_field = FieldSchema(
        name="content_vector", 
        dtype=DataType.FLOAT_VECTOR, 
        dim=1536,  # Assuming using OpenAI embeddings
        description="document content vector"
    )
    
    identitas_field = FieldSchema(
        name="identitas", 
        dtype=DataType.VARCHAR, 
        max_length=65535,
        description="defendant identity information"
    )
    
    riwayat_perkara_field = FieldSchema(
        name="riwayat_perkara",
        dtype=DataType.VARCHAR,
        max_length=65535, 
        description="case history"
    )
    
    amar_putusan_field = FieldSchema(
        name="amar_putusan",
        dtype=DataType.VARCHAR,
        max_length=65535,
        description="court decision"
    )
    
    pertimbangan_hukum_field = FieldSchema(
        name="pertimbangan_hukum",
        dtype=DataType.VARCHAR, 
        max_length=65535,
        description="legal considerations"
    )
    
    url_field = FieldSchema(
        name="url",
        dtype=DataType.VARCHAR,
        max_length=65535,
        description="source document URL"
    )
    
    # Create schema
    schema = CollectionSchema(
        fields=[
            id_field,
            content_vector_field, 
            identitas_field,
            riwayat_perkara_field,
            amar_putusan_field,
            pertimbangan_hukum_field,
            url_field
        ],
        auto_id=False,
        enable_dynamic_field=True,
        description="Collection for legal documents"
    )
    
    return schema

## Create Collection and Index

In [59]:
def create_collection_and_index(collection_name="legal_documents"):
    """Create collection and index"""
    
    # Create schema
    schema = create_legal_document_schema()
    
    # Create collection
    collection = Collection(
        name=collection_name, 
        schema=schema, 
        using='default'
    )
    
    # Create index parameters
    index_params = {
        "index_type": "IVF_FLAT",
        "metric_type": "COSINE",
        "params": {"nlist": 128}
    }
    
    # Create index
    index = Index(collection, "content_vector", index_params)
    
    # Flush collection
    collection.flush()
    
    return collection

## Load Documents to DB

In [60]:
def process_and_insert_documents(documents, embeddings_model, collection_name="legal_documents"):
    """Process documents and insert into collection"""
    
    data_db = []
    
    def embed_text(text):
        """Generate embeddings for text"""
        return embeddings_model.embed_query(text)
    
    # Process documents
    for i, doc in tqdm(enumerate(documents), total=len(documents), desc="Processing Documents"):
        content = doc.page_content
        metadata = doc.metadata
        
        # Parse JSON content
        doc_data = json.loads(content)
        
        # Create combined text for embedding
        combined_text = f"{doc_data.get('identitas', '')} {doc_data.get('riwayat_perkara', '')} {doc_data.get('amar_putusan', '')} {doc_data.get('pertimbangan_hukum', '')}"
        
        # Generate embedding
        content_vector = embed_text(combined_text)
        
        # Create document entry
        data_entry = {
            "id": i,
            "content_vector": content_vector,
            "identitas": doc_data.get('identitas', ''),
            "riwayat_perkara": doc_data.get('riwayat_perkara', ''),
            "amar_putusan": doc_data.get('amar_putusan', ''),
            "pertimbangan_hukum": doc_data.get('pertimbangan_hukum', ''),
            "url": doc_data.get('url', '')
        }
        
        data_db.append(data_entry)
    
    # Get collection
    collection = Collection(collection_name)
    
    # Insert data
    insert_result = collection.insert(data=data_db)
    
    # Flush to ensure data is written
    collection.flush()
    
    # Load
    collection.load()
    
    # Save as JSON backup
    with open('legal_documents_vector.json', 'w') as f:
        json.dump(data_db, f, indent=2)
        
    return insert_result

In [61]:
# Create collection and index
collection = create_collection_and_index()

In [62]:
# Process and insert documents
result = process_and_insert_documents(documents, embeddings)

Processing Documents:  88%|████████▊ | 92/105 [02:39<00:22,  1.73s/it]


KeyboardInterrupt: 

# Retriever

In [20]:
def retrieve_from_milvus(query, collection_name, embeddings, k=5):
    """Retrieve relevant documents from Milvus using hybrid search"""
    
    # Embed the query
    embedded_query = embeddings.embed_query(query)
    collection = Collection(name=collection_name)

    # Set up ANN search parameters
    search_param = {
        "data": [embedded_query],
        "anns_field": "content_vector",
        "param": {
            "metric_type": "COSINE",
            "params": {"nprobe": 10}
        },
        "limit": k
    }
    request = AnnSearchRequest(**search_param)

    # Perform hybrid search
    results = collection.hybrid_search(
        reqs=[request],
        rerank=WeightedRanker(0.8),
        limit=k,
        output_fields=["identitas", "riwayat_perkara", "amar_putusan", "pertimbangan_hukum", "url"]
    )

    # Extract relevant information from results
    documents = []
    if results:
        for hit in results[0]:
            # Access fields using the fields property of the hit object
            fields = hit.fields
            doc_info = {
                "identitas": fields.get("identitas", ""),
                "riwayat_perkara": fields.get("riwayat_perkara", ""),
                "amar_putusan": fields.get("amar_putusan", ""),
                "pertimbangan_hukum": fields.get("pertimbangan_hukum", ""),
                "url": fields.get("url", "")
            }
            documents.append(doc_info)

    return documents

# Example usage:
"""
collection_name = "legal_documents"
query = "apa rangkumannya"
retrieved_docs = retrieve_from_milvus(query, collection_name, embeddings, k=5)
pprint(retrieved_docs)
"""

'\ncollection_name = "legal_documents"\nquery = "apa rangkumannya"\nretrieved_docs = retrieve_from_milvus(query, collection_name, embeddings, k=5)\npprint(retrieved_docs)\n'

In [21]:
from pprint import pprint
collection_name = "legal_documents"
query = "siapa adi daswadi bin wasdam"
retrieved_docs = retrieve_from_milvus(query, collection_name, embeddings, k=5)
pprint(retrieved_docs)

[{'amar_putusan': 'mengadili\n'
                  '1 menyatakan terdakwa adi daswadi bin wasdam tersebut di '
                  'atas terbukti secara sah dan meyakinkan bersalah melakukan '
                  'tindak pidana sebagai penyalahguna narkotika golongan i '
                  'bagi diri sendiri sebagaimana dalam dakwaan alternatif '
                  'kedua\n'
                  '2 menjatuhkan pidana kepada terdakwa oleh karena itu dengan '
                  'pidana penjara selama 2 dua tahun\n'
                  '3 menetapkan masa penangkapan dan penahanan yang telah '
                  'dijalani terdakwa dikurangkan seluruhnya dari pidana yang '
                  'dijatuhkan\n'
                  '4 menetapkan terdakwa tetap ditahan\n'
                  '5 menetapkan barang bukti berupa\n'
                  '1 satu paket narkotika jenis daun ganja kering yang '
                  'dibungkus kertas koran dengan berat bruto kurang lebih 5 40 '
                  'gram setelah dilak

# Generation

In [22]:
# COSTAR-based prompt template
legal_prompt = """
Anda adalah asisten hukum yang dirancang untuk membantu pengguna menganalisis dokumen hukum Indonesia, khususnya putusan pengadilan Mahkamah Agung (MA). Ikuti panduan CO-STAR berikut:

Context:
- Gunakan HANYA informasi dari dokumen putusan pengadilan MA yang disediakan.
- Jika informasi tidak ditemukan dalam dokumen tersebut, jawab "Maaf, saya tidak dapat menemukan informasi tersebut dalam dokumen yang tersedia."
- Fokus pada fakta hukum, seperti identitas terdakwa, riwayat perkara, dan pertimbangan hukum.

Objective:
- Bantu pengguna dalam menganalisis dokumen hukum secara akurat dan efisien menggunakan pendekatan Retrieval-Augmented Generation (RAG).
- Jelaskan terminologi hukum dengan bahasa yang mudah dipahami.
- Jangan memberikan nasihat hukum, tetapi sediakan interpretasi data dari dokumen.
- Gunakan format yang jelas untuk menampilkan hasil analisis.

Style:
- Gunakan gaya bahasa yang formal dan teknis.

Tone:
- Nada profesional dan netral untuk menjaga keakuratan dan kredibilitas respons.

Audience:
- Peneliti hukum, praktisi hukum, atau pengguna umum yang membutuhkan bantuan analisis dokumen putusan MA.

Response:
- Berikan respons dalam format berikut:
  Jawaban: [jawaban]
  URL Sumber: [informasi]
- Jika informasi tidak ditemukan, sebutkan "Tidak tersedia."

Question: {messages}
Context: {context}

Berikan jawaban dalam bahasa Indonesia yang jelas dan terstruktur.
"""


question_answering_prompt = ChatPromptTemplate.from_messages([
    ("system", legal_prompt),
    MessagesPlaceholder(variable_name="messages"),
])

# Document class for handling legal documents
class LegalDocument:
    def __init__(self, page_content, metadata=None):
        self.page_content = page_content
        self.metadata = metadata or {}

# Response generator function
def generate_response(query, llm, collection_name, embeddings):
    """Generate responses to queries using RAG"""
    
    # Initialize chat history
    message_history = ChatMessageHistory()
    
    # Retrieve relevant documents
    retrieved_docs = retrieve_from_milvus(query, collection_name, embeddings, k=5)
    
    # Create document chain
    document_chain = create_stuff_documents_chain(llm, question_answering_prompt)
    
    # Format documents
    docs = [LegalDocument(page_content=str(doc)) for doc in retrieved_docs]
    
    # Add query to message history
    message_history.add_message(HumanMessage(content=query))
    
    # Generate response
    response = document_chain.invoke({
        "context": docs,
        "messages": [HumanMessage(content=query)]
    })
    
    return response

In [32]:
# Initialize necessary components
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI()

# Generate response to a query
query = "siapa sih adi daswadi bin wasdam? dia terlibat kasus apa?"
response = generate_response(
    query=query,
    llm=llm,
    collection_name="legal_documents",
    embeddings=embeddings
)
print(response)

Jawaban: Adi Daswadi bin Wasdam adalah seorang terdakwa dalam sebuah kasus penyalahgunaan narkotika golongan I. Ia terlibat dalam kasus dimana pada tanggal 17 Maret 2018, di dekat stadion Bima, Cirebon, terdakwa ditangkap karena membawa narkotika jenis daun ganja kering. Majelis hakim menyatakan Adi Daswadi bin Wasdam terbukti bersalah dan menjatuhkan pidana penjara selama 2 tahun serta menetapkan agar terdakwa tetap ditahan. Adi Daswadi bin Wasdam juga diminta untuk membayar biaya perkara sejumlah Rp 5.000,00 (lima ribu rupiah).

URL Sumber: https://putusan3.mahkamahagung.go.id/direktori/putusan/0a0e54aa47236640faa91000e271aa0d.html


In [34]:
query = "siapa asep tambun herijadi? dia terlibat kasus apa?"
response = generate_response(
    query=query,
    llm=llm,
    collection_name="legal_documents",
    embeddings=embeddings
)
print(response)

Jawaban: Maaf, saya tidak dapat menemukan informasi tersebut dalam dokumen yang tersedia.
URL Sumber: Tidak tersedia.


# Evaluation